In [1]:
import pandas as pd
import numpy as np

TARGETS = ["y"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
#     [results_1l, results_2l,],
#     ignore_index=True )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_y,MSE_ZZx1_y,R2_ZZx2_y,MSE_ZZx2_y,R2_ZZxReto_y,MSE_ZZxReto_y
0,model_arch1_r0.01_Ld0.5_Lp0.5_seed4080,[1],0.5,0.5,0.01,4080,0.973120,0.922055,0.970488,0.843703,-4.909024,0.076342
1,model_arch1_r0.01_Ld0.5_Lp0.5_seed1926,[1],0.5,0.5,0.01,1926,0.972931,0.923526,0.971375,0.849837,-4.813950,0.082241
2,model_arch1_r0.01_Ld0.5_Lp0.5_seed679,[1],0.5,0.5,0.01,679,0.973275,0.922303,0.971056,0.845237,-4.890363,0.077699
3,model_arch1_r0.01_Ld0.5_Lp0.5_seed5929,[1],0.5,0.5,0.01,5929,0.973075,0.923449,0.971353,0.849124,-4.834555,0.081021
4,model_arch1_r0.01_Ld0.5_Lp0.5_seed896,[1],0.5,0.5,0.01,896,0.973277,0.922309,0.970911,0.844985,-4.882662,0.077867
...,...,...,...,...,...,...,...,...,...,...,...,...
3022,model_arch100_r0.9_Ld0.7_Lp0.3_seed5613,[100],0.7,0.3,0.90,5613,0.997736,0.988890,0.992734,0.938815,0.729385,0.737334
3023,model_arch100_r0.9_Ld0.7_Lp0.3_seed7093,[100],0.7,0.3,0.90,7093,0.984216,0.947880,0.963693,0.890912,-2.761715,0.330269
3024,model_arch100_r0.9_Ld0.7_Lp0.3_seed5756,[100],0.7,0.3,0.90,5756,0.983121,0.947681,0.958506,0.892701,-2.735350,0.334992
3025,model_arch100_r0.9_Ld0.7_Lp0.3_seed4434,[100],0.7,0.3,0.90,4434,0.980727,0.945042,0.963689,0.886753,-3.033254,0.304491


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33


for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"] - 
        0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - y


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
2668,model_arch89_r0.01_Ld0.5_Lp0.5_seed7093,[89],0.998655,0.994510,0.815615,0.916447
2250,model_arch76_r0.01_Ld0.5_Lp0.5_seed4080,[76],0.994326,0.986325,0.818080,0.913629
2843,model_arch94_r0.9_Ld0.7_Lp0.3_seed7093,[94],0.999252,0.987661,0.801238,0.908977
2666,model_arch88_r0.9_Ld0.7_Lp0.3_seed2243,[88],0.999734,0.982150,0.794741,0.904925
2876,model_arch95_r0.9_Ld0.7_Lp0.3_seed2243,[95],0.990209,0.980475,0.803232,0.904867



📊 MÉTRICAS COMPLETAS - TOP 5 (y)


,model,Neurons,R2_ZZx1_y,R2_ZZx2_y,R2_ZZxReto_y,R2_train_mean,R2_val_mean,R2_test_mean,Score
2668,model_arch89_r0.01_Ld0.5_Lp0.5_seed7093,[89],0.998655,0.994510,0.815615,0.998655,0.994510,0.815615,0.916447
2250,model_arch76_r0.01_Ld0.5_Lp0.5_seed4080,[76],0.994326,0.986325,0.818080,0.994326,0.986325,0.818080,0.913629
2843,model_arch94_r0.9_Ld0.7_Lp0.3_seed7093,[94],0.999252,0.987661,0.801238,0.999252,0.987661,0.801238,0.908977
2666,model_arch88_r0.9_Ld0.7_Lp0.3_seed2243,[88],0.999734,0.982150,0.794741,0.999734,0.982150,0.794741,0.904925
2876,model_arch95_r0.9_Ld0.7_Lp0.3_seed2243,[95],0.990209,0.980475,0.803232,0.990209,0.980475,0.803232,0.904867


In [5]:
final_table.to_excel("BestModels-otm.xlsx")

In [6]:
# ============================================
# MÉDIA, DESVIO, MÍNIMO E MÁXIMO
# ============================================

summary_tables = {}

for target in TARGETS:

    top_df = results.copy()
    rows = []

    for s in SETS_CATEGORY.keys():

        r2_col = f"R2_{s.replace('-', '_')}_{target}"
        mse_col = f"MSE_{s.replace('-', '_')}_{target}"

        row = {
            "Set": s,
            "Category": SETS_CATEGORY[s]
        }

        # =========================
        # R²
        # =========================
        if r2_col in top_df.columns:
            row["R2_mean"] = top_df[r2_col].mean()
            row["R2_std"]  = top_df[r2_col].std()
            row["R2_min"]  = top_df[r2_col].min()
            row["R2_max"]  = top_df[r2_col].max()
        else:
            row["R2_mean"] = np.nan
            row["R2_std"]  = np.nan
            row["R2_min"]  = np.nan
            row["R2_max"]  = np.nan

        # =========================
        # MSE
        # =========================
        if mse_col in top_df.columns:
            row["MSE_mean"] = top_df[mse_col].mean()
            row["MSE_std"]  = top_df[mse_col].std()
            row["MSE_min"]  = top_df[mse_col].min()
            row["MSE_max"]  = top_df[mse_col].max()
        else:
            row["MSE_mean"] = np.nan
            row["MSE_std"]  = np.nan
            row["MSE_min"]  = np.nan
            row["MSE_max"]  = np.nan

        rows.append(row)

    summary_df = pd.DataFrame(rows)

    summary_tables[target] = summary_df

    # =========================
    # MOSTRA SOMENTE A TABELA
    # =========================
    display(
        summary_df.style.format({
            "R2_mean": "{:.4f}",
            "R2_std":  "{:.4f}",
            "R2_min":  "{:.4f}",
            "R2_max":  "{:.4f}",

            "MSE_mean": "{:.6f}",
            "MSE_std":  "{:.6f}",
            "MSE_min":  "{:.6f}",
            "MSE_max":  "{:.6f}"
        })
    )

AttributeError: The '.style' accessor requires jinja2